In [6]:
import pandas as pd

df = pd.read_csv("../data/interim/smart_logistics_cleaned.csv")
print(df.columns)


Index(['timestamp', 'asset_id', 'latitude', 'longitude', 'inventory_level',
       'shipment_status', 'temperature', 'humidity', 'traffic_status',
       'waiting_time', 'user_transaction_amount', 'user_purchase_frequency',
       'logistics_delay_reason', 'asset_utilization', 'demand_forecast',
       'logistics_delay', 'month', 'dow', 'hour'],
      dtype='object')


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load processed dataset
df = pd.read_csv("../data/interim/smart_logistics_cleaned.csv")

# Features & target
X = df.drop(columns=["logistics_delay"])   # corrected target column
y = df["logistics_delay"]

# Identify categorical & numeric columns
categorical_cols = X.select_dtypes(include=["object"]).columns
numeric_cols = X.select_dtypes(exclude=["object"]).columns

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

# Logistic Regression pipeline
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(class_weight="balanced", solver="liblinear"))
])

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train
logreg_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = logreg_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, logreg_pipeline.predict_proba(X_test)[:,1]))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        91
           1       1.00      1.00      1.00       109

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

ROC-AUC: 1.0
